## 12_ Champion Model Optimisation (XGBoost)
Explainable AI Credit Risk Decision Platform: Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Reason:
Systematically fine-tunes the hyperparameters and structural parameters of your best-performing model to maximize predictive accuracy.

In [13]:
# IMPORTS & CONFIGURATION
# ________________________________________

from __future__ import annotations

import gc
import logging
import warnings
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sparse

from scipy.stats import randint, uniform

from xgboost import XGBClassifier


from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

PROJECT_NAME = "Explainable AI Credit Risk Decision Platform"

NOTEBOOK_NAME = "12_champion_model_optimisation"

PROJECT_ROOT = Path.cwd()

FEATURE_STORE_DIR = PROJECT_ROOT / "data" / "feature_store"

MODEL_DIR = PROJECT_ROOT / "models"

REPORT_DIR = PROJECT_ROOT / "reports"

METRIC_DIR = REPORT_DIR / "metrics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

METRIC_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_NAME)
print("=" * 50)
print(f"Notebook : {NOTEBOOK_NAME}")
print(f"Execution: {datetime.now():%Y-%m-%d %H:%M}")

Explainable AI Credit Risk Decision Platform
Notebook : 12_champion_model_optimisation
Execution: 2026-07-21 11:08


In [4]:
# REUSABLE HELPER FUNCTIONS
# _____________________________________________

# logger
# ________________________

def get_logger():

    logger = logging.getLogger("ChampionModel")

    if not logger.handlers:

        logger.setLevel(logging.INFO)

        handler = logging.StreamHandler()

        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)s | %(message)s")

        handler.setFormatter(formatter)

        logger.addHandler(handler)

    return logger

logger = get_logger()

In [49]:
# Load matrix
#___________________

def load_sparse_matrix(filename):

    logger.info(f"Loading {filename}")

    matrix = sparse.load_npz(FEATURE_STORE_DIR / filename)

    logger.info("Loaded successfully.")

    return matrix
    
# Load Target
#_______________________

def load_target():

    y_train = np.load(FEATURE_STORE_DIR / "y_train.npy")

    y_test = np.load(FEATURE_STORE_DIR / "y_test.npy")

    return y_train, y_test
    
# Save model
# _______________________

def save_model(model, filename):

    path = MODEL_DIR / filename

    joblib.dump(model, path)

    logger.info(f"Saved: {filename}")

In [50]:
# Model Evaluation
# _____________________________________

def evaluate_model(model, X_test, y_test):

    prediction = model.predict(X_test)

    probability = model.predict_proba(X_test)[:, 1]

    return pd.DataFrame({

        "Accuracy":[accuracy_score(y_test, prediction)],

        "Precision":[precision_score(y_test, prediction)],

        "Recall":[recall_score(y_test, prediction)],

        "F1":[f1_score(y_test, prediction)],

        "ROC_AUC":[roc_auc_score(y_test, probability)]})

# Clear Memory
#_________________________________________

def clear_memory(*objects):

    for obj in objects:
        del obj

    gc.collect()

    logger.info("Memory cleared.")

### Load Champion Model Dataset
##### ___________________________________
##### This notebook loads only Matrix D,

##### because it was selected as the champion feature set.

In [51]:
# LOAD CHAMPION DATASET
# _________________________________________

logger.info("Loading Champion Dataset")

matrix_D_train = load_sparse_matrix("matrix_D_train.npz")

matrix_D_test = load_sparse_matrix("matrix_D_test.npz")

y_train, y_test = load_target()

print()

print("Training Shape :", matrix_D_train.shape)

print("Testing Shape  :", matrix_D_test.shape)

print("Target Train   :", y_train.shape)

print("Target Test    :", y_test.shape)

2026-07-21 14:54:57,808 | INFO | Loading Champion Dataset
2026-07-21 14:54:57,810 | INFO | Loading matrix_D_train.npz
2026-07-21 14:54:58,269 | INFO | Loaded successfully.
2026-07-21 14:54:58,284 | INFO | Loading matrix_D_test.npz
2026-07-21 14:54:58,343 | INFO | Loaded successfully.



Training Shape : (400000, 91)
Testing Shape  : (100000, 91)
Target Train   : (400000,)
Target Test    : (100000,)


In [52]:
# VALIDATION
# ____________________________

assert matrix_D_train.shape[0] == y_train.shape[0]

assert matrix_D_test.shape[0] == y_test.shape[0]

logger.info("Dataset validation passed.")

2026-07-21 14:55:04,513 | INFO | Dataset validation passed.


In [53]:
# DEFINE HYPERPARAMETER SEARCH SPACE
# ___________________________________________________________

print("Define Hyperparameter Search Space")

logger.info("Building XGBoost hyperparameter search space...")

param_grid = {

    "n_estimators": randint(100, 500),

    "learning_rate": uniform(0.01, 0.19),

    "max_depth": randint(3, 10),

    "min_child_weight": randint(1, 10),

    "subsample": uniform(0.6, 0.4),

    "colsample_bytree": uniform(0.6, 0.4),

    "gamma": uniform(0.0, 5.0),

    "reg_alpha": uniform(0.0, 2.0),

    "reg_lambda": uniform(0.5, 3.0)}

logger.info(f"Total Hyperparameters : {len(param_grid)}")

display(pd.DataFrame({"Hyperparameter": param_grid.keys(),
                      
    "Distribution": [str(v) for v in param_grid.values()]}))

2026-07-21 14:55:13,924 | INFO | Building XGBoost hyperparameter search space...
2026-07-21 14:55:13,935 | INFO | Total Hyperparameters : 9


Define Hyperparameter Search Space


,Hyperparameter,Distribution
0,n_estimators,<scipy.stats._distn_infrastructure.rv_discrete...
1,learning_rate,<scipy.stats._distn_infrastructure.rv_continuo...
2,max_depth,<scipy.stats._distn_infrastructure.rv_discrete...
3,min_child_weight,<scipy.stats._distn_infrastructure.rv_discrete...
4,subsample,<scipy.stats._distn_infrastructure.rv_continuo...
5,colsample_bytree,<scipy.stats._distn_infrastructure.rv_continuo...
6,gamma,<scipy.stats._distn_infrastructure.rv_continuo...
7,reg_alpha,<scipy.stats._distn_infrastructure.rv_continuo...
8,reg_lambda,<scipy.stats._distn_infrastructure.rv_continuo...


In [54]:
# TIME-BASED CROSS VALIDATION
# ________________________________________________

print("Time-Based Cross Validation")

logger.info("Creating TimeSeriesSplit...")

tscv = TimeSeriesSplit(n_splits=5)

logger.info(f"Number of folds : {tscv.get_n_splits()}")

print(tscv)

2026-07-21 14:55:21,327 | INFO | Creating TimeSeriesSplit...
2026-07-21 14:55:21,329 | INFO | Number of folds : 5


Time-Based Cross Validation
TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [55]:
# RANDOMIZED SEARCH
# _________________________________________________

print("Randomized Hyperparameter Optimisation")

negative = (y_train == 0).sum()

positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

baseline_model = XGBClassifier(

    objective="binary:logistic",

    eval_metric="logloss",

    random_state=RANDOM_STATE,

    tree_method="hist",

    n_jobs=-1,

    scale_pos_weight=scale_pos_weight)

random_search = RandomizedSearchCV(

    estimator=baseline_model,

    param_distributions=param_grid,

    n_iter=30,

    scoring="roc_auc",

    cv=tscv,

    verbose=2,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    refit=True)

logger.info("Starting optimisation...")

random_search.fit(

    matrix_D_train,

    y_train)

logger.info("Optimisation completed.")

2026-07-21 14:55:26,992 | INFO | Starting optimisation...


Randomized Hyperparameter Optimisation
Fitting 5 folds for each of 30 candidates, totalling 150 fits


2026-07-21 15:15:46,954 | INFO | Optimisation completed.


In [57]:
# BEST PARAMETERS
# ____________________________________________

best_params = random_search.best_params_

best_score = random_search.best_score_

print("\nBest ROC-AUC (CV):", round(best_score,4))

display(pd.DataFrame({

    "Parameter": best_params.keys(),

    "Best Value": best_params.values()}))


Best ROC-AUC (CV): 0.7216


,Parameter,Best Value
0,colsample_bytree,0.824511
1,gamma,3.854836
2,learning_rate,0.103821
3,max_depth,3.000000
4,min_child_weight,3.000000
5,n_estimators,162.000000
6,reg_alpha,1.791527
7,reg_lambda,1.926111
8,subsample,0.825310


In [59]:
# TRAIN OPTIMISED XGBOOST
# _____________________________________________

print("Train Optimised Champion Model")

best_model = random_search.best_estimator_

best_model.fit(matrix_D_train,y_train)

save_model(best_model,"champion_xgboost.pkl")

logger.info("Champion model trained and saved.")

Train Optimised Champion Model
[CV] END colsample_bytree=0.7334834444556088, gamma=0.7143340896097039, learning_rate=0.13366880986028204, max_depth=7, min_child_weight=2, n_estimators=443, reg_alpha=1.6648852816008435, reg_lambda=1.1370173320348285, subsample=0.6727299868828402; total time=  31.9s
[CV] END colsample_bytree=0.9895022075365837, gamma=1.1638567021515211, learning_rate=0.02721522256123595, max_depth=8, min_child_weight=3, n_estimators=463, reg_alpha=1.0284688768272232, reg_lambda=2.2772437065861273, subsample=0.6185801650879991; total time=  35.7s
[CV] END colsample_bytree=0.8430179407605753, gamma=0.8526206184364576, learning_rate=0.02235980266720311, max_depth=6, min_child_weight=9, n_estimators=415, reg_alpha=1.1265764356910786, reg_lambda=1.6562495076197483, subsample=0.6063865008880857; total time=  56.0s
[CV] END colsample_bytree=0.9314950036607718, gamma=1.7837666334679465, learning_rate=0.06337755684060234, max_depth=6, min_child_weight=9, n_estimators=256, reg_alp

2026-07-21 15:20:28,429 | INFO | Saved: champion_xgboost.pkl
2026-07-21 15:20:28,430 | INFO | Champion model trained and saved.


In [64]:
# PERFORMANCE COMPARISON
# ________________________________________________

print("Performance Comparison")

optimised_results = evaluate_model(

    best_model,

    matrix_D_test,

    y_test)

optimised_results.insert(

    0,

    "Model",

    "Optimised XGBoost")

baseline_results = pd.read_csv(

    METRIC_DIR / "xgboost_results.csv")

baseline_results = (baseline_results[
    
    baseline_results["Dataset"]=="Matrix D"].copy())

baseline_results.insert(

    0,

    "Model",

    "Baseline XGBoost")

optimised_results.insert(
    1,
    
    "Dataset",
    
    "Matrix D")

comparison = pd.concat([baseline_results,
                        
                        optimised_results],
                       
                       ignore_index=True)

display(comparison)

Performance Comparison


,Model,Dataset,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline XGBoost,Matrix D,0.63508,0.266932,0.704602,0.387183,0.724106
1,Optimised XGBoost,Matrix D,0.63182,0.266072,0.711081,0.387245,0.724671


In [65]:
# PERFORMANCE IMPROVEMENT
# ________________________________________

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

improvement = pd.DataFrame({

    "Metric": metrics,

    "Baseline": baseline_results.iloc[0][metrics].values,

    "Optimised": optimised_results.iloc[0][metrics].values})

improvement["Difference"] = (

    improvement["Optimised"]

    -

    improvement["Baseline"])

display(improvement.round(4))

,Metric,Baseline,Optimised,Difference
0,Accuracy,0.63508,0.63182,-0.00326
1,Precision,0.266932,0.266072,-0.00086
2,Recall,0.704602,0.711081,0.006479
3,F1,0.387183,0.387245,0.000062
4,ROC_AUC,0.724106,0.724671,0.000565


In [70]:
# EXPORT MODEL EVALUATION ARTEFACTS
# ___________________________________________________

print("Export Model Evaluation Artefacts")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Comparison Table

comparison.to_csv(REPORT_DIR / "model_comparison.csv",index=False)

# Best Model Only

champion_results = comparison.loc[comparison["Model"] == "Optimised XGBoost"]

champion_results.to_csv(REPORT_DIR / "champion_model_results.csv",index=False)

joblib.dump(best_model,MODEL_DIR / "optimised_xgboost.pkl")

# Performance Summary

summary = comparison.describe()

summary.to_csv(REPORT_DIR / "model_performance_summary.csv")

logger.info("Model evaluation artefacts exported successfully.")

2026-07-22 12:22:14,033 | INFO | Model evaluation artefacts exported successfully.


Export Model Evaluation Artefacts
